# Unsupervised Clustering Analysis — MTSK Corpus

**Complementary analysis to:** *An Interpretable LLM-Based Classifier for Research Articles on Mathematics Teaching Specialized Knowledge*

**Authors:** Carolina Rojas Celis, Luz Maricel Elorreaga Rodríguez, Héctor Javier Hortúa  
**Institution:** Universidad El Bosque — Maestría en Estadística Aplicada y Ciencia de Datos

---

## Purpose

This notebook performs an **exploratory unsupervised clustering analysis** of the MTSK corpus (293 documents in Spanish, English, and Portuguese) to characterize its latent thematic structure **independently of the supervised classification**.

**Important note:** This analysis uses `paraphrase-multilingual-MiniLM-L12-v2` for generating semantic embeddings — a different model from the supervised classifier (`intfloat/multilingual-e5-large`). This is intentional: the purpose is to describe the natural organization of the corpus, not to analyze the classifier's internal representations. The choice of model does not affect the validity of the conclusions.

## Pipeline

```
Raw corpus (293 docs) → SBERT embeddings (384d) → PCA (50d) → UMAP (5d) → KMeans clustering
                                                             → UMAP (2d) → Visualization
```

## Environment

Designed for **Google Colab**. Run cells in order. Restart kernel after cell 1.

## Reproducibility

All random seeds are fixed (`random_state=42`). Library versions are pinned (see Cell 1).  
Pre-computed embeddings can be saved and reloaded to avoid recalculation (Cell 14).

In [ ]:
# ============================================================
# CELL 1 — Install libraries (pinned versions for reproducibility)
# After running this cell, RESTART the runtime before continuing
# Runtime > Restart runtime  (or the button that appears below)
# ============================================================
!pip install -q sentence-transformers==2.7.0
!pip install -q datasets==2.14.0
!pip install -q umap-learn==0.5.6
!pip install -q scikit-learn==1.3.2

print("✅ Installation complete. Please RESTART the runtime before continuing.")

In [ ]:
# ============================================================
# CELL 2 — Imports
# ============================================================
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import warnings
warnings.filterwarnings('ignore')

from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA
from sklearn.preprocessing import normalize
from sklearn.metrics import (silhouette_score, davies_bouldin_score,
                              calinski_harabasz_score, adjusted_rand_score,
                              normalized_mutual_info_score)
from sklearn.metrics.pairwise import cosine_similarity
from scipy.cluster.hierarchy import dendrogram, linkage
from scipy.optimize import linear_sum_assignment
import umap

# Fixed seed for reproducibility
SEED = 42
np.random.seed(SEED)

print("✅ Libraries loaded")

In [ ]:
# ============================================================
# CELL 3 — Load dataset
# Mount Google Drive and load the MTSK corpus
# Expected columns: 'Título', 'Resumen (texto literal)', 'Palabras clave (texto literal)'
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# ── Update this path to match your Drive location ──
DATA_PATH = '/content/drive/MyDrive/TG Maestria/No supervisado/DATACEROMTSK_AUMENTADO_SE.xlsx'

df = pd.read_excel(DATA_PATH)
print(f"Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")
print(f"\nExpected: 293 documents")
df.head(3)

In [ ]:
# ============================================================
# CELL 4 — Preprocessing: combine text fields
# Title and keywords are duplicated to give them higher weight
# relative to the abstract.
# ============================================================
def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r'\n', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['titulo_c']   = df['Título'].apply(clean_text)
df['resumen_c']  = df['Resumen (texto literal)'].apply(clean_text)
df['keywords_c'] = df['Palabras clave (texto literal)'].apply(clean_text)

# Title × 2, keywords × 2 for implicit weighting
df['texto_completo'] = (
    df['titulo_c']   + '. ' + df['titulo_c']   + '. ' +
    df['resumen_c']  + '. ' +
    df['keywords_c'] + '. ' + df['keywords_c']
)

empty = df['texto_completo'].str.strip().eq('').sum()
print(f"Empty texts: {empty}")
print(f"Total documents: {len(df)}")
print("\nExample:")
print(df['texto_completo'].iloc[0][:300])

In [ ]:
# ============================================================
# CELL 5 — Multilingual embeddings with SBERT
#
# Model: paraphrase-multilingual-MiniLM-L12-v2
# Selected for computational efficiency in exploratory analysis.
# Note: this differs from the supervised classifier (intfloat/multilingual-e5-large).
# The purpose here is to characterize the corpus structure,
# not to reproduce the classifier's internal representations.
# ============================================================
print("Loading multilingual SBERT model...")
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Generating embeddings (1-2 minutes)...")
textos = df['texto_completo'].tolist()
embeddings = model.encode(textos, show_progress_bar=True, batch_size=32)

print(f"\n✅ Embeddings generated: {embeddings.shape}")
# Expected: (293, 384) — 293 articles × 384 dimensions

In [ ]:
# ============================================================
# CELL 6 — Dimensionality reduction: PCA → UMAP
#
# PCA (384 → 50): removes noise, retains 85.5% of variance
# UMAP 5D (50 → 5): for clustering (preserves more structure than 2D)
# UMAP 2D (50 → 2): for visualization only
# ============================================================
# Normalize
X = normalize(embeddings)

# PCA: 50 components
pca = PCA(n_components=50, random_state=SEED)
X_pca = pca.fit_transform(X)
var_acum = np.cumsum(pca.explained_variance_ratio_)
print(f"Cumulative variance with 50 PCA components: {var_acum[-1]*100:.1f}%")
for n in [10, 20, 30, 50]:
    print(f"  {n} components → {var_acum[n-1]*100:.1f}%")

# UMAP 2D — visualization
print("\nComputing UMAP 2D...")
reducer_2d = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1,
                        metric='cosine', random_state=SEED)
X_2d = reducer_2d.fit_transform(X_pca)

# UMAP 5D — clustering
print("Computing UMAP 5D...")
reducer_5d = umap.UMAP(n_components=5, n_neighbors=15, min_dist=0.05,
                        metric='cosine', random_state=SEED)
X_5d = reducer_5d.fit_transform(X_pca)

print(f"\n✅ Reduction complete:")
print(f"   PCA:     {X_pca.shape}")
print(f"   UMAP 2D: {X_2d.shape}")
print(f"   UMAP 5D: {X_5d.shape}")

In [ ]:
# ============================================================
# CELL 7 — Optimal K selection: elbow + Silhouette
# Clustering is performed on UMAP 5D (richer representation than 2D)
# ============================================================
ks = range(2, 11)
inertias, silhouettes = [], []

for k in ks:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    labels = km.fit_predict(X_5d)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_5d, labels))

k_opt = list(ks)[np.argmax(silhouettes)]
print(f"Optimal K by Silhouette: {k_opt}")
print("\nScores by K:")
for k, s in zip(ks, silhouettes):
    marker = " ← optimal" if k == k_opt else ""
    print(f"  K={k}: silhouette={s:.4f}{marker}")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Optimal number of clusters selection', fontsize=14, fontweight='bold')

axes[0].plot(list(ks), inertias, 'o-', color='#E63946', linewidth=2.5, markersize=8)
axes[0].axvline(x=k_opt, color='#457B9D', linestyle='--', alpha=0.7, label=f'Optimal K = {k_opt}')
axes[0].set_title('Elbow Method')
axes[0].set_xlabel('Number of clusters K')
axes[0].set_ylabel('Inertia')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

bars = axes[1].bar(list(ks), silhouettes,
                   color=['#E63946' if k == k_opt else '#A8DADC' for k in ks],
                   edgecolor='white')
for bar, val in zip(bars, silhouettes):
    axes[1].text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.005,
                 f'{val:.3f}', ha='center', va='bottom', fontsize=9, fontweight='bold')
axes[1].set_title('Silhouette Score by K')
axes[1].set_xlabel('Number of clusters K')
axes[1].set_ylabel('Silhouette Score')
axes[1].axhline(y=0.5, color='#2A9D8F', linestyle='--', alpha=0.6, label='Acceptable threshold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('seleccion_k.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 8 — Apply clustering algorithms
# KMeans, Hierarchical, GMM with optimal K and K=5 (supervised reference)
# DBSCAN for outlier detection
# ============================================================
resultados = {}

for k in [k_opt, 5]:
    km = KMeans(n_clusters=k, random_state=SEED, n_init=10)
    resultados[f'KMeans_k{k}']      = km.fit_predict(X_5d)
    agg = AgglomerativeClustering(n_clusters=k)
    resultados[f'Hierarchical_k{k}'] = agg.fit_predict(X_5d)
    gmm = GaussianMixture(n_components=k, random_state=SEED)
    resultados[f'GMM_k{k}']         = gmm.fit_predict(X_5d)

# DBSCAN — thematic outlier detection
db = DBSCAN(eps=0.5, min_samples=5)
resultados['DBSCAN'] = db.fit_predict(X_5d)
n_clusters_db = len(set(resultados['DBSCAN'])) - (1 if -1 in resultados['DBSCAN'] else 0)
n_outliers    = (resultados['DBSCAN'] == -1).sum()
print(f"DBSCAN → {n_clusters_db} clusters, {n_outliers} outliers")

# Metrics table
print(f"\n{'Model':<25} {'Silhouette ↑':>13} {'Davies-Bouldin ↓':>17} {'Calinski-H ↑':>13}")
print("─" * 70)
for nombre, labels in resultados.items():
    mask = labels != -1
    if mask.sum() < 10 or len(set(labels[mask])) < 2:
        continue
    s   = silhouette_score(X_5d[mask], labels[mask])
    dbs = davies_bouldin_score(X_5d[mask], labels[mask])
    ch  = calinski_harabasz_score(X_5d[mask], labels[mask])
    print(f"{nombre:<25} {s:>13.4f} {dbs:>17.4f} {ch:>13.2f}")

In [ ]:
# ============================================================
# CELL 9 — UMAP 2D visualization of all clustering models
# ============================================================
paletas = {
    3: ['#1D3557', '#E63946', '#2A9D8F'],
    5: ['#1D3557', '#E63946', '#2A9D8F', '#F4A261', '#6A0572'],
}

fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('MTSK Article Clustering — UMAP 2D Projection\n(Multilingual SBERT Embeddings)',
             fontsize=16, fontweight='bold', color='#1D3557')

modelos_plot = [
    (f'KMeans K={k_opt} (Optimal)',    resultados[f'KMeans_k{k_opt}'],    k_opt),
    ('KMeans K=5 (supervised reference)', resultados['KMeans_k5'],           5),
    ('Hierarchical K=5',                  resultados['Hierarchical_k5'],     5),
    ('DBSCAN — Thematic outliers',        resultados['DBSCAN'],             -1),
]

for ax, (titulo, labels, k) in zip(axes.flat, modelos_plot):
    if k == -1:  # DBSCAN
        unique = sorted(set(labels))
        cmap = plt.cm.tab20(np.linspace(0, 1, max(len(unique), 1)))
        for i, lbl in enumerate(unique):
            mask = labels == lbl
            if lbl == -1:
                ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c='#CCCCCC',
                           label=f'Outliers (n={mask.sum()})', alpha=0.5, s=40, marker='x')
            else:
                ax.scatter(X_2d[mask, 0], X_2d[mask, 1], color=cmap[i],
                           label=f'C{lbl+1} (n={mask.sum()})', alpha=0.8, s=55,
                           edgecolors='white', linewidths=0.4)
    else:
        colores = paletas.get(k, paletas[5])
        for c in range(k):
            mask = labels == c
            ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=colores[c],
                       label=f'Group {c+1} (n={mask.sum()})',
                       alpha=0.85, s=60, edgecolors='white', linewidths=0.4)
        # Mark centroids
        for c in range(k):
            mask = labels == c
            cx, cy = X_2d[mask, 0].mean(), X_2d[mask, 1].mean()
            ax.scatter(cx, cy, color=colores[c], s=250, marker='*',
                       edgecolors='black', linewidths=1, zorder=5)

    ax.set_title(titulo, fontsize=12, fontweight='bold', color='#1D3557')
    ax.set_xlabel('Semantic dimension 1', fontsize=10)
    ax.set_ylabel('Semantic dimension 2', fontsize=10)
    ax.legend(fontsize=9, framealpha=0.9)
    ax.grid(True, alpha=0.2)
    ax.set_facecolor('#FAFAFA')

plt.tight_layout()
plt.savefig('clustering_umap2d.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 10 — Hierarchical dendrogram
# ============================================================
fig, ax = plt.subplots(figsize=(16, 6))
Z = linkage(X_5d, method='ward')
dendrogram(Z, ax=ax, truncate_mode='level', p=6,
           color_threshold=Z[-(k_opt), 2],
           above_threshold_color='#AAAAAA',
           leaf_font_size=8)
ax.set_title('Dendrogram — Hierarchical Clustering of MTSK Articles',
             fontsize=14, fontweight='bold', color='#1D3557')
ax.set_xlabel('Articles', fontsize=11)
ax.set_ylabel('Distance (Ward)', fontsize=11)
ax.axhline(y=Z[-(k_opt), 2], color='#E63946', linestyle='--', alpha=0.8,
           label=f'Cut K={k_opt} (optimal)')
ax.axhline(y=Z[-5, 2], color='#F4A261', linestyle='--', alpha=0.8,
           label='Cut K=5 (supervised reference)')
ax.legend(fontsize=11)
ax.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.savefig('dendrograma.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ============================================================
# CELL 11 — Semantic interpretation of clusters
# Most representative articles per cluster (cosine similarity to centroid)
# ============================================================
k_analizar = k_opt
labels_analizar = resultados[f'KMeans_k{k_analizar}']
df['cluster'] = labels_analizar

print(f"{'='*70}")
print(f"SEMANTIC INTERPRETATION — K={k_analizar}")
print(f"{'='*70}\n")

for c in range(k_analizar):
    mask = labels_analizar == c
    subset = df[mask].reset_index(drop=True)
    centroide = embeddings[mask].mean(axis=0, keepdims=True)
    sims = cosine_similarity(centroide, embeddings[mask])[0]
    top3 = sims.argsort()[-3:][::-1]

    print(f"{'─'*70}")
    print(f"GROUP {c+1}  |  {mask.sum()} articles  ({mask.sum()/len(df)*100:.1f}%)")
    print(f"{'─'*70}")
    print("Most representative articles:")
    for i in top3:
        titulo = str(subset.loc[i, 'Título']).replace('\n', ' ').strip()
        print(f"  ⭐ {titulo[:85]}")
    print()

In [ ]:
# ============================================================
# CELL 12 — Outlier analysis (DBSCAN)
# Articles that do not fit any thematic group
# ============================================================
outliers_mask = resultados['DBSCAN'] == -1
df_outliers = df[outliers_mask][['Título', 'Resumen (texto literal)']].copy()

print(f"OUTLIER ARTICLES — {outliers_mask.sum()} atypical documents")
print("These articles do not fit well in any thematic cluster\n")
for i, row in df_outliers.iterrows():
    titulo = str(row['Título']).replace('\n', ' ').strip()
    print(f"  [{i}] {titulo[:90]}")

In [ ]:
# ============================================================
# CELL 13 — Agreement metrics with supervised classification
# Requires: supervised labels in the original dataset
# Column name may vary — update col_tema if needed
# ============================================================

# Path to dataset WITH supervised labels
SUP_PATH = '/content/drive/MyDrive/TG Maestria/DATACEROMTSK_AUMENTADO.xlsx'

df_sup = pd.read_excel(SUP_PATH)

# Auto-detect thematic column
col_tema = [c for c in df_sup.columns if 'lasif' in c or 'ematica' in c or 'opic' in c]
print(f"Thematic column detected: {col_tema}")
col_tema = col_tema[0]

def limpiar(t):
    return re.sub(r'\s+', ' ', str(t).strip().lower())

df_sup['titulo_key'] = df_sup['Título'].apply(limpiar)
df['titulo_key']     = df['Título'].apply(limpiar)

df_merged = df.merge(df_sup[['titulo_key', col_tema]], on='titulo_key', how='left')
df_merged = df_merged.rename(columns={col_tema: 'Tematica_Supervisado'})
df_merged = df_merged.dropna(subset=['Tematica_Supervisado']).drop_duplicates(subset='titulo_key').reset_index(drop=True)
df_merged['T_num'] = df_merged['Tematica_Supervisado'].map({'T1':1,'T2':2,'T3':3,'T4':4,'T5':5})

labels_sup = df_merged['T_num'].values
labels_k3  = resultados[f'KMeans_k{k_opt}'][:len(df_merged)]
labels_k5  = resultados['KMeans_k5'][:len(df_merged)]

ari3 = adjusted_rand_score(labels_sup, labels_k3)
nmi3 = normalized_mutual_info_score(labels_sup, labels_k3)
ari5 = adjusted_rand_score(labels_sup, labels_k5)
nmi5 = normalized_mutual_info_score(labels_sup, labels_k5)

print("\n" + "="*55)
print("AGREEMENT METRICS — Supervised vs Unsupervised")
print("="*55)
print(f"\n{'Metric':<35} {'K=3':>8} {'K=5':>8}")
print("-" * 55)
print(f"{'Adjusted Rand Index (ARI)':<35} {ari3:>8.4f} {ari5:>8.4f}")
print(f"{'Normalized Mutual Info (NMI)':<35} {nmi3:>8.4f} {nmi5:>8.4f}")
print("\nNote: Low ARI is expected — supervised classification uses expert")
print("theoretical criteria; unsupervised uses surface lexical signals only.")
print("Divergence is informative, not an error.")

# Thematic composition heatmap
import matplotlib
matplotlib.rcParams.update({'font.size': 13})

mat3     = pd.crosstab(resultados[f'KMeans_k{k_opt}'][:len(df_merged)]+1, df_merged['Tematica_Supervisado'])
mat3_pct = (mat3.div(mat3.sum(axis=1), axis=0) * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
vals = mat3_pct.values
im   = ax.imshow(vals, cmap='Blues', aspect='auto', vmin=0, vmax=55)
ax.set_xticks(range(5))
ax.set_xticklabels(['T1','T2','T3','T4','T5'], fontsize=13, fontweight='bold')
ax.set_yticks(range(k_opt))
ax.set_yticklabels([f'G{i+1}' for i in range(k_opt)], fontsize=13, fontweight='bold')
ax.set_xlabel('Supervised thematic classification', fontsize=13, labelpad=10)
ax.set_ylabel('Automatic group', fontsize=13, labelpad=10)
ax.set_title('Thematic composition by automatic group\n(% of articles per theme within each group)',
             fontsize=13, fontweight='bold', color='#1D3557', pad=14)
umbral = vals.max() * 0.55
for i in range(k_opt):
    for j in range(5):
        v = vals[i, j]
        ax.text(j, i, f'{v:.1f}%', ha='center', va='center', fontsize=12, fontweight='bold',
                color='white' if v > umbral else '#1D3557')
plt.colorbar(im, ax=ax, shrink=0.85, pad=0.02, label='% within group')
plt.tight_layout()
plt.savefig('figura_composicion_tematica.png', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print("✅ Saved: figura_composicion_tematica.png")

In [ ]:
# ============================================================
# CELL 14 — Export results and save embeddings
# ============================================================
df_export = df[['Título', 'Resumen (texto literal)', 'Palabras clave (texto literal)']].copy()
df_export[f'Cluster_K{k_opt}'] = resultados[f'KMeans_k{k_opt}'] + 1
df_export['Cluster_K5']         = resultados['KMeans_k5'] + 1
df_export['DBSCAN_label']       = resultados['DBSCAN']
df_export['Is_outlier']         = resultados['DBSCAN'] == -1

df_export.to_excel('resultados_clustering_MTSK.xlsx', index=False)
print("✅ Exported: resultados_clustering_MTSK.xlsx")
print(f"\nDistribution K={k_opt}:")
print(df_export[f'Cluster_K{k_opt}'].value_counts().sort_index())
print(f"\nDistribution K=5:")
print(df_export['Cluster_K5'].value_counts().sort_index())

# Save embeddings for reuse (avoids recalculating)
np.save('embeddings_MTSK_multilingual.npy', embeddings)
np.save('X_umap_2d.npy', X_2d)
np.save('X_umap_5d.npy', X_5d)
print("\n✅ Embeddings saved:")
print(f"   embeddings_MTSK_multilingual.npy → {embeddings.shape}")
print(f"   X_umap_2d.npy                    → {X_2d.shape}")
print(f"   X_umap_5d.npy                    → {X_5d.shape}")
print("\nNote: .npy files are in .gitignore — do not commit to GitHub.")